# 05b — Stacking Full Pool (ZIT family + reg_only + two_stage_grid + NN)

**목적**: 가용 base 전부 모아서 ElasticNetCV meta-learner 로 stacking. NN 베이스가 plateau 깨는지 검증.

**Base pool (≤27)**:
| 그룹 | 모델 | 출처 |
|---|---|---|
| ZIT family (6) | zit_only, bag_zit_combined_best(_xy), bag_zit_pp_hpo, bag_zit_hpo, bag_zit_fixed_ge | final/zit_only, _temp/bag_zit_* |
| reg_only (4) | lgbm, enet, et, catboost | final/reg_only/* |
| two_stage_grid (16) | {clf}_x_{reg} for clf∈{catboost,et,lgbm,xgb}, reg∈{catboost,enet,et,xgb} | final/two_stage_grid/combined/* |
| NN (1) | nn_ft | final/nn_ft |

**Meta-learner**: ElasticNetCV(StandardScaler), 동일 하이퍼파라미터 그리드 (`alpha=logspace(-6,0,30)`, `l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0]`).

**비교 baseline**:
- 단일 best base val
- Blending SLSQP (Σw=1, w≥0)
- 기존 stacking_11base val=0.005701

**격리**: `4_output/final/stacking_full_with_nn/` 신규. 모듈 무수정.

**선행 조건**: 모든 base 의 `oof_unit.csv`, `val_unit.csv`, `test_unit.csv` 가 존재해야 함. 누락된 모델은 자동 제외.

## 1. 환경 + import

In [1]:
import os, sys, json

%run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all

from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from scipy.optimize import minimize

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. Base pool 정의 + 자동 탐지

각 디렉토리에 3개 파일 (`oof_unit.csv`, `val_unit.csv`, `test_unit.csv`) 모두 존재해야 pool 포함. 누락 모델은 자동 제외.

In [2]:
POOL_PATHS = {}

# 1) ZIT family 6종
POOL_PATHS['zit_only'] = os.path.join(OUTPUT_DIR, 'final', 'zit_only')
for n in ['bag_zit_combined_best', 'bag_zit_combined_best_xy',
         'bag_zit_pp_hpo', 'bag_zit_hpo', 'bag_zit_fixed_ge']:
    POOL_PATHS[n] = os.path.join(OUTPUT_DIR, '_temp', n)

# 2) reg_only 4종
for n in ['lgbm', 'enet', 'et', 'catboost']:
    POOL_PATHS[f'reg__{n}'] = os.path.join(OUTPUT_DIR, 'final', 'reg_only', n)

# 3) two_stage_grid 16조합 (4 clf × 4 reg)
_combined_dir = os.path.join(OUTPUT_DIR, 'final', 'two_stage_grid', 'combined')
for clf in ['catboost', 'et', 'lgbm', 'xgb']:
    for reg in ['catboost', 'enet', 'et', 'xgb']:
        POOL_PATHS[f'grid__{clf}_x_{reg}'] = os.path.join(_combined_dir, f'{clf}_x_{reg}')

# 4) NN 1종
POOL_PATHS['nn_ft'] = os.path.join(OUTPUT_DIR, 'final', 'nn_ft')

REQUIRED = ['oof_unit.csv', 'val_unit.csv', 'test_unit.csv']
available, missing = {}, {}
for name, base in POOL_PATHS.items():
    have = all(os.path.exists(os.path.join(base, f)) for f in REQUIRED)
    if have:
        available[name] = base
    else:
        miss = [f for f in REQUIRED if not os.path.exists(os.path.join(base, f))]
        missing[name] = miss

print(f'=== Pool 자동 탐지 ===')
print(f'  사용 가능 ({len(available)}/{len(POOL_PATHS)}):')
for n in available:
    print(f'    ✓ {n}')
if missing:
    print(f'\n  누락 ({len(missing)}):')
    for n, miss in missing.items():
        print(f'    ✗ {n}  (missing: {miss})')

if len(available) < 2:
    raise RuntimeError(f'stacking 가능한 base < 2개')

=== Pool 자동 탐지 ===
  사용 가능 (27/27):
    ✓ zit_only
    ✓ bag_zit_combined_best
    ✓ bag_zit_combined_best_xy
    ✓ bag_zit_pp_hpo
    ✓ bag_zit_hpo
    ✓ bag_zit_fixed_ge
    ✓ reg__lgbm
    ✓ reg__enet
    ✓ reg__et
    ✓ reg__catboost
    ✓ grid__catboost_x_catboost
    ✓ grid__catboost_x_enet
    ✓ grid__catboost_x_et
    ✓ grid__catboost_x_xgb
    ✓ grid__et_x_catboost
    ✓ grid__et_x_enet
    ✓ grid__et_x_et
    ✓ grid__et_x_xgb
    ✓ grid__lgbm_x_catboost
    ✓ grid__lgbm_x_enet
    ✓ grid__lgbm_x_et
    ✓ grid__lgbm_x_xgb
    ✓ grid__xgb_x_catboost
    ✓ grid__xgb_x_enet
    ✓ grid__xgb_x_et
    ✓ grid__xgb_x_xgb
    ✓ nn_ft


## 3. OOF/val/test 로드 + 정합 검증

각 csv는 `[ufs_serial, pred, health]` (또는 `[ufs_serial, prob, pred, health]`) 컬럼. `pred` 만 사용. 모든 base 동일 ufs_serial set.

In [3]:
_, ys = load_all()

# csv 내장 health 사용 (CLIP_Y_EXTREME 적용된 첫번째 base 기준)
first_name = next(iter(available))
first_oof = pd.read_csv(os.path.join(available[first_name], 'oof_unit.csv'))
first_val = pd.read_csv(os.path.join(available[first_name], 'val_unit.csv'))
first_test = pd.read_csv(os.path.join(available[first_name], 'test_unit.csv'))
y_oof  = first_oof.set_index(KEY_COL)['health']
y_val  = first_val.set_index(KEY_COL)['health']
y_test = first_test.set_index(KEY_COL)['health']

print(f'y counts: oof={len(y_oof):,}, val={len(y_val):,}, test={len(y_test):,}')
print(f'y max (clipped): oof={y_oof.max():.6f}, val={y_val.max():.6f}, test={y_test.max():.6f}')

# 모든 base 모델 pred 로드
oofs, vals, tests = {}, {}, {}
for n, base in available.items():
    oofs[n]  = pd.read_csv(os.path.join(base, 'oof_unit.csv')).set_index(KEY_COL)['pred']
    vals[n]  = pd.read_csv(os.path.join(base, 'val_unit.csv')).set_index(KEY_COL)['pred']
    tests[n] = pd.read_csv(os.path.join(base, 'test_unit.csv')).set_index(KEY_COL)['pred']

# 정합 검증 — 1개라도 idx mismatch 면 base 자동 drop (warn)
def _check_index(d, ref, label):
    drop = []
    for n, s in d.items():
        miss = ref.difference(s.index)
        extra = s.index.difference(ref)
        if len(miss) or len(extra):
            print(f'  [WARN] {label}/{n}: missing={len(miss)}, extra={len(extra)} → drop')
            drop.append(n)
    return drop

_drop = set()
_drop |= set(_check_index(oofs,  y_oof.index,  'oof'))
_drop |= set(_check_index(vals,  y_val.index,  'val'))
_drop |= set(_check_index(tests, y_test.index, 'test'))

if _drop:
    print(f'\n  index mismatch 로 drop: {sorted(_drop)}')
    for n in _drop:
        oofs.pop(n, None); vals.pop(n, None); tests.pop(n, None)
        available.pop(n, None)

# 정렬된 prediction matrix
P_oof  = pd.DataFrame({n: oofs[n].reindex(y_oof.index)   for n in available})
P_val  = pd.DataFrame({n: vals[n].reindex(y_val.index)   for n in available})
P_test = pd.DataFrame({n: tests[n].reindex(y_test.index) for n in available})

# NaN 검사
for label, df in [('oof', P_oof), ('val', P_val), ('test', P_test)]:
    n_nan = df.isna().sum().sum()
    if n_nan > 0:
        print(f'  [WARN] {label}: NaN {n_nan}개 발견')

print(f'\n[정합 OK] P_oof={P_oof.shape}, P_val={P_val.shape}, P_test={P_test.shape}')
print(f'  최종 base 수: {len(available)}')

[load_xs] all-NaN 행 407개 제거 → 174,573행


[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572


[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
y counts: oof=26,187, val=8,727, test=8,729
y max (clipped): oof=1.000000, val=0.172211, test=0.602242



[정합 OK] P_oof=(26187, 27), P_val=(8727, 27), P_test=(8729, 27)
  최종 base 수: 27


## 4. 단일 base RMSE + residual correlation

27개라 corr matrix 가 크므로 **NN 과 다른 base 의 residual corr** 만 별도 정렬해서 출력.

In [4]:
def _rmse(p, y):
    return float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))

rows = []
for n in available:
    rows.append({
        'model': n,
        'oof':   _rmse(P_oof[n].values,  y_oof.values),
        'val':   _rmse(P_val[n].values,  y_val.values),
        'test':  _rmse(P_test[n].values, y_test.values),
    })
single_df = pd.DataFrame(rows).sort_values('val').reset_index(drop=True)
print(f'=== 단일 base RMSE (val 오름차순, top {len(single_df)}) ===')
print(single_df.to_string(index=False, float_format='%.6f'))

# residual corr
R_oof  = P_oof.subtract(y_oof,  axis=0)
R_test = P_test.subtract(y_test, axis=0)
C_oof  = R_oof.corr()
C_test = R_test.corr()

# NN 과의 잔차 상관 (오름차순)
if 'nn_ft' in available:
    nn_corr_oof  = C_oof['nn_ft'].drop('nn_ft').sort_values()
    nn_corr_test = C_test['nn_ft'].drop('nn_ft').sort_values()
    print(f'\n=== NN vs 다른 base — residual corr (OOF, low → high) ===')
    print(nn_corr_oof.round(4).to_string())

# 가장 보완적인 페어 (OOF)
names_list = list(available)
min_pair = (None, None, 1.0)
for i, a in enumerate(names_list):
    for b in names_list[i+1:]:
        c = C_oof.loc[a, b]
        if c < min_pair[2]:
            min_pair = (a, b, c)
print(f'\n가장 낮은 OOF residual corr: {min_pair[0]} ↔ {min_pair[1]} = {min_pair[2]:.4f}')

=== 단일 base RMSE (val 오름차순, top 27) ===
                    model      oof      val     test
                 zit_only 0.008246 0.005709 0.008414
    bag_zit_combined_best 0.008251 0.005710 0.008412
              bag_zit_hpo 0.008244 0.005711 0.008417
 bag_zit_combined_best_xy 0.008251 0.005712 0.008411
           bag_zit_pp_hpo 0.008245 0.005712 0.008414
         bag_zit_fixed_ge 0.008253 0.005717 0.008412
            reg__catboost 0.008259 0.005731 0.008430
                reg__lgbm 0.008257 0.005731 0.008429
grid__catboost_x_catboost 0.008298 0.005739 0.008429
     grid__catboost_x_xgb 0.008290 0.005742 0.008431
           grid__xgb_x_et 0.008328 0.005751 0.008433
         grid__xgb_x_enet 0.008328 0.005752 0.008434
      grid__catboost_x_et 0.008307 0.005753 0.008421
    grid__catboost_x_enet 0.008308 0.005754 0.008422
                  reg__et 0.008269 0.005758 0.008452
     grid__xgb_x_catboost 0.008333 0.005771 0.008460
          grid__xgb_x_xgb 0.008325 0.005777 0.008464
      

## 5. Blending baseline (SLSQP — Σw=1, w≥0)

Stacking 효과 비교용. 27 base 동일 가중치 시작 → SLSQP 최적화.

In [5]:
K = len(available)
P_oof_arr  = P_oof.values
P_val_arr  = P_val.values
P_test_arr = P_test.values
y_oof_arr  = y_oof.values
y_val_arr  = y_val.values
y_test_arr = y_test.values

res = minimize(
    lambda w: _rmse(P_oof_arr @ w, y_oof_arr),
    np.full(K, 1.0/K),
    method='SLSQP',
    bounds=[(0.0, 1.0)] * K,
    constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
    options={'ftol': 1e-9, 'maxiter': 1000},
)
w_blend = res.x

blend_oof  = P_oof_arr  @ w_blend
blend_val  = P_val_arr  @ w_blend
blend_test = P_test_arr @ w_blend

rmse_blend_oof  = _rmse(blend_oof,  y_oof_arr)
rmse_blend_val  = _rmse(blend_val,  y_val_arr)
rmse_blend_test = _rmse(blend_test, y_test_arr)

print(f'=== Blending SLSQP (converged={res.success}) — top 10 weight ===')
for n, w in sorted(zip(available, w_blend), key=lambda x: -x[1])[:10]:
    bar = '█' * int(w * 60)
    print(f'  {n:32s}: {w:.4f}  {bar}')
n_zero = sum(1 for w in w_blend if w < 1e-6)
print(f'  ... 0-weight base: {n_zero}/{K}')
print(f'\n  OOF RMSE:  {rmse_blend_oof:.6f}')
print(f'  val RMSE:  {rmse_blend_val:.6f}')
print(f'  test RMSE: {rmse_blend_test:.6f}')

=== Blending SLSQP (converged=True) — top 10 weight ===
  bag_zit_fixed_ge                : 0.2667  ███████████████
  zit_only                        : 0.1981  ███████████
  bag_zit_hpo                     : 0.1768  ██████████
  bag_zit_pp_hpo                  : 0.1358  ████████
  bag_zit_combined_best           : 0.0705  ████
  bag_zit_combined_best_xy        : 0.0621  ███
  grid__et_x_et                   : 0.0356  ██
  grid__et_x_enet                 : 0.0336  ██
  grid__catboost_x_enet           : 0.0078  
  reg__lgbm                       : 0.0066  
  ... 0-weight base: 14/27

  OOF RMSE:  0.008241
  val RMSE:  0.005706
  test RMSE: 0.008408


## 6. Stacking — ElasticNetCV meta + StandardScaler

**ElasticNetCV** 5-fold CV로 (alpha, l1_ratio) 자동 선택. l1_ratio 0.9~1.0 이면 Lasso 성격이 강해 redundant base 제거. 음수 weight 허용 (corrector).

In [6]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('enet',   ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        alphas=np.logspace(-6, 0, 30),
        cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
        random_state=SEED,
        n_jobs=-1,
        max_iter=20000,
        positive=False,
    )),
])
pipe.fit(P_oof_arr, y_oof_arr)

en = pipe.named_steps['enet']
best_alpha    = float(en.alpha_)
best_l1_ratio = float(en.l1_ratio_)
stack_coef    = en.coef_
stack_intercept = float(en.intercept_)

print(f'=== ElasticNetCV best ===')
print(f'  alpha    : {best_alpha:.6e}')
print(f'  l1_ratio : {best_l1_ratio:.4f}')
print(f'  intercept: {stack_intercept:+.6f}')
print(f'\n  coef (scaled space, abs 정렬, 비영 항만):')
for n, c in sorted(zip(available, stack_coef), key=lambda x: -abs(x[1])):
    if abs(c) < 1e-9:
        continue
    bar = '█' * int(min(abs(c) * 1500, 50))
    sign = '+' if c >= 0 else '-'
    print(f'    {n:32s}: {sign}{abs(c):.5f}  {bar}')
n_active = sum(1 for c in stack_coef if abs(c) > 1e-9)
n_neg    = sum(1 for c in stack_coef if c < -1e-9)
print(f'  활성 base: {n_active}/{K} (음수 weight {n_neg}개 = corrector)')

stack_oof  = np.clip(pipe.predict(P_oof_arr),  0, None)
stack_val  = np.clip(pipe.predict(P_val_arr),  0, None)
stack_test = np.clip(pipe.predict(P_test_arr), 0, None)

rmse_stack_oof  = _rmse(stack_oof,  y_oof_arr)
rmse_stack_val  = _rmse(stack_val,  y_val_arr)
rmse_stack_test = _rmse(stack_test, y_test_arr)

neg_pre_clip_oof  = (pipe.predict(P_oof_arr)  < 0).mean()
neg_pre_clip_val  = (pipe.predict(P_val_arr)  < 0).mean()
neg_pre_clip_test = (pipe.predict(P_test_arr) < 0).mean()

print(f'\n=== Stacking RMSE ===')
print(f'  OOF RMSE:  {rmse_stack_oof:.6f}')
print(f'  val RMSE:  {rmse_stack_val:.6f}')
print(f'  test RMSE: {rmse_stack_test:.6f}')
print(f'  음수 clip 비율: oof={neg_pre_clip_oof:.1%}, val={neg_pre_clip_val:.1%}, test={neg_pre_clip_test:.1%}')

# NN 의 weight 별도 출력
if 'nn_ft' in available:
    nn_idx = list(available).index('nn_ft')
    nn_coef = stack_coef[nn_idx]
    print(f'\n  [NN] coef = {nn_coef:+.6f}  ({"활성" if abs(nn_coef) > 1e-9 else "0 — 풀에 흡수"})')

=== ElasticNetCV best ===
  alpha    : 4.520354e-05
  l1_ratio : 0.5000
  intercept: +0.002515

  coef (scaled space, abs 정렬, 비영 항만):
    bag_zit_hpo                     : +0.00056  
    bag_zit_fixed_ge                : +0.00040  
    zit_only                        : +0.00021  
    grid__et_x_catboost             : -0.00007  
    grid__lgbm_x_xgb                : -0.00006  
    grid__et_x_xgb                  : -0.00005  
    grid__et_x_et                   : +0.00004  
    nn_ft                           : +0.00004  
    reg__enet                       : -0.00002  
    grid__xgb_x_catboost            : -0.00001  
  활성 base: 10/27 (음수 weight 5개 = corrector)

=== Stacking RMSE ===
  OOF RMSE:  0.008238
  val RMSE:  0.005705
  test RMSE: 0.008407
  음수 clip 비율: oof=0.0%, val=0.0%, test=0.0%

  [NN] coef = +0.000036  (활성)


## 7. 종합 비교 표

In [7]:
best_oof_row  = single_df.sort_values('oof').iloc[0]
best_val_row  = single_df.sort_values('val').iloc[0]
best_test_row = single_df.sort_values('test').iloc[0]

# 기존 stacking_11base val (참고용)
STACKING_11BASE_VAL = 0.005701
STACKING_11BASE_TEST = 0.008408

comparison = pd.DataFrame([
    {'method': f'best single (OOF) [{best_oof_row["model"]}]',
     'oof': best_oof_row['oof'], 'val': best_oof_row['val'], 'test': best_oof_row['test']},
    {'method': f'best single (val) [{best_val_row["model"]}]',
     'oof': best_val_row['oof'], 'val': best_val_row['val'], 'test': best_val_row['test']},
    {'method': f'stacking_11base (기존)',
     'oof': float('nan'), 'val': STACKING_11BASE_VAL, 'test': STACKING_11BASE_TEST},
    {'method': f'Blending SLSQP ({K} base)',
     'oof': rmse_blend_oof, 'val': rmse_blend_val, 'test': rmse_blend_test},
    {'method': f'Stacking ElasticNet ({K} base)',
     'oof': rmse_stack_oof, 'val': rmse_stack_val, 'test': rmse_stack_test},
])
print('=' * 100)
print(f'  Stacking Full Pool — 종합 비교 (사용 가능 base = {K}종)')
print('=' * 100)
print(comparison.to_string(index=False, float_format='%.6f'))
print('=' * 100)

# Δ
print('\n  Stacking 효과:')
for label, oof, val, test in [
    ('Blending  vs single best       ', rmse_blend_oof, rmse_blend_val, rmse_blend_test),
    ('Stacking  vs single best       ', rmse_stack_oof, rmse_stack_val, rmse_stack_test),
]:
    d_val  = val  - best_val_row['val']
    d_test = test - best_test_row['test']
    print(f'  {label}: Δval={d_val:+.6f}  Δtest={d_test:+.6f}')

# vs 기존 stacking_11base
d_val_vs_11  = rmse_stack_val  - STACKING_11BASE_VAL
d_test_vs_11 = rmse_stack_test - STACKING_11BASE_TEST
print(f'\n  Stacking  vs stacking_11base   : Δval={d_val_vs_11:+.6f}  Δtest={d_test_vs_11:+.6f}')
if d_val_vs_11 < 0:
    print(f'  → NN/grid 추가가 plateau 깸 (val ↓{abs(d_val_vs_11):.6f})')
elif d_val_vs_11 > 0.00005:
    print(f'  → NN/grid 추가가 오히려 손해 (over-pool noise)')
else:
    print(f'  → plateau 그대로 (val 변화 미미)')

  Stacking Full Pool — 종합 비교 (사용 가능 base = 27종)
                         method      oof      val     test
best single (OOF) [bag_zit_hpo] 0.008244 0.005711 0.008417
   best single (val) [zit_only] 0.008246 0.005709 0.008414
           stacking_11base (기존)      NaN 0.005701 0.008408
       Blending SLSQP (27 base) 0.008241 0.005706 0.008408
  Stacking ElasticNet (27 base) 0.008238 0.005705 0.008407

  Stacking 효과:
  Blending  vs single best       : Δval=-0.000002  Δtest=-0.000003
  Stacking  vs single best       : Δval=-0.000004  Δtest=-0.000004

  Stacking  vs stacking_11base   : Δval=+0.000004  Δtest=-0.000001
  → plateau 그대로 (val 변화 미미)


## 8. 산출물 저장 (`4_output/final/stacking_full_with_nn/`)

In [8]:
OUT_DIR = os.path.join(OUTPUT_DIR, 'final', 'stacking_full_with_nn')
os.makedirs(OUT_DIR, exist_ok=True)

def _save_unit(pred_arr, ids, y_arr, fname):
    out = pd.DataFrame({
        KEY_COL: ids,
        'pred':  pred_arr,
        'health': y_arr.reindex(ids).values,
    })
    out.to_csv(os.path.join(OUT_DIR, fname), index=False)

# stacking 산출물
_save_unit(stack_oof,  y_oof.index,  y_oof,  'oof_unit_stack.csv')
_save_unit(stack_val,  y_val.index,  y_val,  'val_unit_stack.csv')
_save_unit(stack_test, y_test.index, y_test, 'test_unit_stack.csv')

# blending baseline 산출물
_save_unit(blend_oof,  y_oof.index,  y_oof,  'oof_unit_blend.csv')
_save_unit(blend_val,  y_val.index,  y_val,  'val_unit_blend.csv')
_save_unit(blend_test, y_test.index, y_test, 'test_unit_blend.csv')

single_df.to_csv(os.path.join(OUT_DIR, 'single_base_rmse.csv'), index=False)
C_oof.to_csv(os.path.join(OUT_DIR, 'residual_corr_oof.csv'))
C_test.to_csv(os.path.join(OUT_DIR, 'residual_corr_test.csv'))
comparison.to_csv(os.path.join(OUT_DIR, 'comparison.csv'), index=False)

meta = {
    'pool_models':     list(available.keys()),
    'pool_paths':      {k: available[k] for k in available},
    'missing_models':  list(missing.keys()),
    'meta_learner': {
        'type':       'ElasticNetCV(StandardScaler)',
        'alpha':      best_alpha,
        'l1_ratio':   best_l1_ratio,
        'coef':       {n: float(c) for n, c in zip(available, stack_coef)},
        'intercept':  stack_intercept,
        'positive':   False,
        'max_iter':   20000,
        'cv':         '5-fold KFold(shuffle=True, random_state=SEED)',
        'n_active':   int(n_active),
        'n_negative': int(n_neg),
    },
    'blending_slsqp': {
        'weights':    {n: float(w) for n, w in zip(available, w_blend)},
        'converged':  bool(res.success),
        'rmse_oof':   rmse_blend_oof,
        'rmse_val':   rmse_blend_val,
        'rmse_test':  rmse_blend_test,
    },
    'stacking': {
        'rmse_oof':  rmse_stack_oof,
        'rmse_val':  rmse_stack_val,
        'rmse_test': rmse_stack_test,
    },
    'baseline_stacking_11base': {
        'rmse_val':  STACKING_11BASE_VAL,
        'rmse_test': STACKING_11BASE_TEST,
    },
    'delta_vs_11base': {
        'val':  rmse_stack_val  - STACKING_11BASE_VAL,
        'test': rmse_stack_test - STACKING_11BASE_TEST,
    },
    'single_base': single_df.to_dict(orient='records'),
    'min_residual_corr_pair': {
        'a': min_pair[0], 'b': min_pair[1], 'corr_oof': float(min_pair[2]),
    },
    'SEED': int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

저장 완료: C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\stacking_full_with_nn
  comparison.csv                         0.4 KB
  meta.json                             11.4 KB
  oof_unit_blend.csv                   970.9 KB
  oof_unit_stack.csv                   971.2 KB
  residual_corr_oof.csv                 13.9 KB
  residual_corr_test.csv                13.9 KB
  single_base_rmse.csv                   2.1 KB
  test_unit_blend.csv                  323.6 KB
  test_unit_stack.csv                  323.7 KB
  val_unit_blend.csv                   323.6 KB
  val_unit_stack.csv                   323.7 KB


## 9. 요약

In [9]:
print('=' * 100)
print(f' Stacking Full Pool ({K}-base) — 결과 요약')
print('=' * 100)
print(comparison.to_string(index=False, float_format='%.6f'))
print('-' * 100)
print(f'  Meta best alpha    : {best_alpha:.4e}')
print(f'  Meta best l1_ratio : {best_l1_ratio:.4f}')
print(f'  Meta intercept     : {stack_intercept:+.6f}')
print(f'  활성 base          : {n_active}/{K} (음수 weight {n_neg}개 = corrector)')
print(f'  최저 residual corr : {min_pair[0]} ↔ {min_pair[1]} = {min_pair[2]:.4f}')
if 'nn_ft' in available:
    nn_idx = list(available).index('nn_ft')
    nn_coef = stack_coef[nn_idx]
    nn_min_corr = C_oof['nn_ft'].drop('nn_ft').min()
    print(f'  NN coef            : {nn_coef:+.6f}  ({"활성" if abs(nn_coef) > 1e-9 else "흡수됨"})')
    print(f'  NN 최저 잔차 corr  : {nn_min_corr:.4f}')
print('=' * 100)
print(f'  → NN/grid 추가가 stacking_11base 대비 Δval={rmse_stack_val - STACKING_11BASE_VAL:+.6f}, '
      f'Δtest={rmse_stack_test - STACKING_11BASE_TEST:+.6f}')
print(f'  → NN coef ≈ 0 이면 plateau 흡수 → paradigm 다양성 부족 결론')

 Stacking Full Pool (27-base) — 결과 요약
                         method      oof      val     test
best single (OOF) [bag_zit_hpo] 0.008244 0.005711 0.008417
   best single (val) [zit_only] 0.008246 0.005709 0.008414
           stacking_11base (기존)      NaN 0.005701 0.008408
       Blending SLSQP (27 base) 0.008241 0.005706 0.008408
  Stacking ElasticNet (27 base) 0.008238 0.005705 0.008407
----------------------------------------------------------------------------------------------------
  Meta best alpha    : 4.5204e-05
  Meta best l1_ratio : 0.5000
  Meta intercept     : +0.002515
  활성 base          : 10/27 (음수 weight 5개 = corrector)
  최저 residual corr : grid__lgbm_x_enet ↔ nn_ft = 0.9694
  NN coef            : +0.000036  (활성)
  NN 최저 잔차 corr  : 0.9694
  → NN/grid 추가가 stacking_11base 대비 Δval=+0.000004, Δtest=-0.000001
  → NN coef ≈ 0 이면 plateau 흡수 → paradigm 다양성 부족 결론
